# 02 · ETL e Integração SIH + CNES

**Objetivo:** Carregar os dados brutos do SIH, selecionar colunas relevantes, filtrar internações IAM, limpar e integrar com CNES.

**Fluxo:**
```
1. Selecionar colunas        → 113 → 37–56 colunas
2. Filtrar CID I21           → somente IAM
3. Remover < 18 anos         → adultos apenas
4. Tratar valores ausentes   → base limpa
5. Integrar com CNES         → base_modelagem.parquet
```

## 0. Configurações

In [4]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW_SIH   = Path('data/input/SIH')
RAW_CNES  = Path('data/input/CNES')
INTERIM   = Path('data/interim')
PROCESSED = Path('data/processed')
EXTERNAL  = Path('data/external')

for p in [INTERIM, PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

SEP      = ';'
ENCODING = 'latin-1'

---
## 1. Seleção de colunas

O SIH possui 113 colunas. A maioria é **administrativa** (controle de faturamento, dados de processo) ou **específica de outros grupos** (obstétrico, trabalhador). Para IAM em adultos, mantemos apenas o que tem relação com desfecho, diagnóstico, gravidade ou infraestrutura.

**Critério de descarte:**
- Campo administrativo sem valor preditivo (ex: CPF autorizador, sequência de remessa)
- Campo populações específicas — vazio em IAM (ex: CONTRACEP, GESTRISCO, INSTRU)
- Componentes financeiros redundantes com VAL_TOT
- Componentes parciais de UTI redundantes com UTI_INT_TO e UTI_MES_TO

In [5]:
# ── Colunas a MANTER (37) ─────────────────────────────────────────────────────
COLUNAS_MANTER = [
    # Chave
    'N_AIH', 'CNES',
    # Variável-alvo
    'MORTE',
    # Tempo — sobrevivência
    'DT_INTER', 'DT_SAIDA', 'DIAS_PERM', 'ANO_CMPT', 'MES_CMPT',
    # Paciente
    'NASC', 'IDADE', 'COD_IDADE', 'SEXO', 'RACA_COR', 'NACIONAL',
    # Diagnóstico
    'DIAG_PRINC', 'DIAG_SECUN', 'CID_ASSO', 'CID_MORTE',
    'DIAGSEC1', 'DIAGSEC2', 'DIAGSEC3', 'DIAGSEC4', 'DIAGSEC5', 'DIAGSEC6',
    # Internação
    'ESPEC', 'CAR_INT', 'COMPLEX', 'PROC_SOLIC', 'PROC_REA', 'INFEHOSP',
    # UTI
    'UTI_MES_TO', 'UTI_INT_TO', 'MARCA_UTI',
    # Financeiro (proxies de complexidade)
    'VAL_TOT', 'VAL_UTI',
    # Geo
    'MUNIC_RES', 'MUNIC_MOV',
]

# ── Colunas a AVALIAR — mantemos e decidimos pelo % preenchido nos dados IAM ──
# (serão avaliadas na seção de missing values)
COLUNAS_AVALIAR = [
    'DIAGSEC7', 'DIAGSEC8', 'DIAGSEC9',
    'TPDISEC1', 'TPDISEC2', 'TPDISEC3', 'TPDISEC4',
    'TPDISEC5', 'TPDISEC6', 'TPDISEC7', 'TPDISEC8', 'TPDISEC9',
    'VAL_SH', 'VAL_SP',
    'FINANC', 'FAEC_TP', 'REGCT',
    'NAT_JUR',
    'QT_DIARIAS',
]

COLUNAS_CARREGAR = COLUNAS_MANTER + COLUNAS_AVALIAR
print(f'Colunas a carregar: {len(COLUNAS_CARREGAR)} de 113')
print(f'  MANTER  : {len(COLUNAS_MANTER)}')
print(f'  AVALIAR : {len(COLUNAS_AVALIAR)}')
print(f'  IGNORAR : {113 - len(COLUNAS_CARREGAR)}')

Colunas a carregar: 56 de 113
  MANTER  : 37
  AVALIAR : 19
  IGNORAR : 57


---
## 2. Carregamento do SIH

In [6]:
%%time

def carregar_sih(pasta: Path, colunas: list, sep: str, encoding: str) -> pd.DataFrame:
    """Lê todos os CSVs do SIH carregando apenas as colunas necessárias."""
    arquivos = sorted(glob.glob(str(pasta / '*.csv')))
    if not arquivos:
        raise FileNotFoundError(f'Nenhum CSV encontrado em {pasta}')

    chunks = []
    for arq in arquivos:
        df = pd.read_csv(
            arq, sep=sep, encoding=encoding,
            dtype=str, low_memory=False,
            usecols=lambda c: c in colunas,  # só lê o necessário
        )
        df['_origem'] = Path(arq).name
        chunks.append(df)
        print(f'  {Path(arq).name}: {len(df):,} linhas')

    return pd.concat(chunks, ignore_index=True)

sih_raw = carregar_sih(RAW_SIH, COLUNAS_CARREGAR, SEP, ENCODING)
print(f'\n→ SIH carregado: {sih_raw.shape[0]:,} linhas × {sih_raw.shape[1]} colunas')

FileNotFoundError: Nenhum CSV encontrado em data/input/SIH

---
## 3. Filtro IAM — CID I21

In [8]:
n_total = len(sih_raw)

# Filtra DIAG_PRINC começando com I21 (cobre I21, I21.0, I210 etc.)
mask_iam = sih_raw['DIAG_PRINC'].str.strip().str.upper().str.startswith('I21', na=False)
sih_iam = sih_raw[mask_iam].copy()

print(f'Total registros SIH     : {n_total:>10,}')
print(f'Internações IAM (I21)   : {len(sih_iam):>10,}  ({len(sih_iam)/n_total*100:.2f}%)')
print(f'\nSubcódigos encontrados:')
print(sih_iam['DIAG_PRINC'].value_counts().to_string())

NameError: name 'sih_raw' is not defined

---
## 4. Filtro de idade — somente adultos (≥ 18 anos)

In [ ]:
# COD_IDADE define a unidade de IDADE:
#   2 = meses, 3 = anos (< 1 ano), 4 = anos (≥ 1 ano), 5 = anos (≥ 100 anos)
# Só considera adultos com COD_IDADE == '4' ou '5' E IDADE >= 18

sih_iam['IDADE_num'] = pd.to_numeric(sih_iam['IDADE'], errors='coerce')

print('Distribuição COD_IDADE nos registros IAM:')
print(sih_iam['COD_IDADE'].value_counts(dropna=False).to_string())

n_antes = len(sih_iam)

# Mantém somente idades em anos (COD_IDADE 4 ou 5) e >= 18
mask_adulto = (
    sih_iam['COD_IDADE'].isin(['4', '5']) &
    (sih_iam['IDADE_num'] >= 18)
)
sih_iam = sih_iam[mask_adulto].copy()

n_removidos = n_antes - len(sih_iam)
print(f'\nRemovidos (< 18 anos ou unidade não-anual): {n_removidos:,}')
print(f'Registros restantes                        : {len(sih_iam):,}')
print(f'\nIdade após filtro:')
print(sih_iam['IDADE_num'].describe().round(1).to_string())

---
## 5. Tipagem e limpeza básica

In [ ]:
# ── Variável-alvo ─────────────────────────────────────────────────────────────
sih_iam['MORTE'] = pd.to_numeric(sih_iam['MORTE'], errors='coerce').fillna(0).astype(int)
assert sih_iam['MORTE'].isin([0, 1]).all(), 'MORTE contém valores além de 0 e 1'

# ── Datas ─────────────────────────────────────────────────────────────────────
for col in ['DT_INTER', 'DT_SAIDA']:
    sih_iam[col] = pd.to_datetime(sih_iam[col], format='%Y%m%d', errors='coerce')

# Recalcula dias de internação (mais confiável que DIAS_PERM do DATASUS)
sih_iam['dias_internacao'] = (sih_iam['DT_SAIDA'] - sih_iam['DT_INTER']).dt.days

# Remove internações com datas inválidas ou absurdas
n_antes = len(sih_iam)
sih_iam = sih_iam[sih_iam['dias_internacao'].between(0, 365)].copy()
print(f'Removidos por dias_internacao inválido: {n_antes - len(sih_iam):,}')

# ── Numéricos ─────────────────────────────────────────────────────────────────
for col in ['DIAS_PERM', 'UTI_MES_TO', 'UTI_INT_TO', 'VAL_TOT', 'VAL_UTI',
            'VAL_SH', 'VAL_SP', 'QT_DIARIAS']:
    if col in sih_iam.columns:
        sih_iam[col] = pd.to_numeric(sih_iam[col], errors='coerce')

# ── CNES padronizado (7 dígitos com zeros à esquerda) ─────────────────────────
sih_iam['CNES'] = sih_iam['CNES'].astype(str).str.strip().str.zfill(7)

# ── Derivadas úteis ───────────────────────────────────────────────────────────
sih_iam['usou_uti']  = (sih_iam['UTI_INT_TO'].fillna(0) > 0).astype(int)
sih_iam['ano_inter'] = sih_iam['DT_INTER'].dt.year
sih_iam['mes_inter'] = sih_iam['DT_INTER'].dt.month

print('✓ Tipagem concluída')
print(f'  Shape: {sih_iam.shape}')
print(f'  Taxa de óbito: {sih_iam["MORTE"].mean():.2%}')

In [ ]:
# ── Remove duplicatas por N_AIH ───────────────────────────────────────────────
n_antes = len(sih_iam)
sih_iam = sih_iam.drop_duplicates(subset='N_AIH', keep='first')
print(f'Duplicatas removidas: {n_antes - len(sih_iam):,}')
print(f'Registros únicos    : {len(sih_iam):,}')

---
## 6. Avaliação de missing values e decisão sobre colunas AVALIAR

In [ ]:
# ── % missing por coluna nos dados IAM ───────────────────────────────────────
missing = sih_iam.isnull().mean().mul(100).round(1).sort_values(ascending=False)
missing_df = missing.reset_index()
missing_df.columns = ['coluna', 'pct_missing']

# Classifica por grupo
def grupo(col):
    if col in ['DIAGSEC7','DIAGSEC8','DIAGSEC9']: return 'AVALIAR - diag secundário'
    if col.startswith('TPDISEC'):                  return 'AVALIAR - tipo diag sec'
    if col in ['VAL_SH','VAL_SP']:                 return 'AVALIAR - financeiro'
    if col in ['FINANC','FAEC_TP','REGCT']:        return 'AVALIAR - gestão'
    if col == 'NAT_JUR':                           return 'AVALIAR - hospital'
    if col == 'QT_DIARIAS':                        return 'AVALIAR - internação'
    return 'MANTER'

missing_df['grupo'] = missing_df['coluna'].apply(grupo)

print('=== COLUNAS EM AVALIAÇÃO — % missing nos dados IAM ===')
avaliar_miss = missing_df[missing_df['grupo'].str.startswith('AVALIAR')]
print(avaliar_miss.to_string(index=False))

print('\n=== COLUNAS MANTER — % missing nos dados IAM ===')
manter_miss = missing_df[missing_df['grupo'] == 'MANTER']
print(manter_miss[manter_miss['pct_missing'] > 0].to_string(index=False))

In [ ]:
# ── Decisão automática: descarta colunas AVALIAR com > 50% missing ────────────
LIMIAR_MISSING = 50.0

cols_avaliar_missing = missing_df[
    missing_df['grupo'].str.startswith('AVALIAR')
][['coluna', 'pct_missing']]

descartar_por_missing = cols_avaliar_missing[
    cols_avaliar_missing['pct_missing'] > LIMIAR_MISSING
]['coluna'].tolist()

manter_apos_avaliacao = cols_avaliar_missing[
    cols_avaliar_missing['pct_missing'] <= LIMIAR_MISSING
]['coluna'].tolist()

print(f'Limiar de missing: {LIMIAR_MISSING}%')
print(f'\nDescartadas (>{LIMIAR_MISSING}% missing): {len(descartar_por_missing)}')
for c in descartar_por_missing:
    pct = cols_avaliar_missing[cols_avaliar_missing['coluna']==c]['pct_missing'].values[0]
    print(f'  {c:<20} {pct:.1f}%')

print(f'\nMantidas (<={LIMIAR_MISSING}% missing): {len(manter_apos_avaliacao)}')
for c in manter_apos_avaliacao:
    pct = cols_avaliar_missing[cols_avaliar_missing['coluna']==c]['pct_missing'].values[0]
    print(f'  {c:<20} {pct:.1f}%')

sih_iam = sih_iam.drop(columns=descartar_por_missing, errors='ignore')
print(f'\nShape após descarte por missing: {sih_iam.shape}')

---
## 7. Tratamento de valores ausentes nas colunas mantidas

In [ ]:
# ── Estratégia por tipo de coluna ─────────────────────────────────────────────
#
# UTI: 0 é o valor natural quando paciente não foi para UTI — não é missing real
# Diagnósticos secundários: '0000' ou ausente = sem comorbidade registrada
# Financeiros numéricos: 0 quando não aplicável
# Categóricos: 'NAO_INF' para preservar a ausência como categoria

# UTI — missing = não usou UTI = 0
for col in ['UTI_MES_TO', 'UTI_INT_TO', 'VAL_UTI']:
    if col in sih_iam.columns:
        sih_iam[col] = sih_iam[col].fillna(0)

# Financeiros — missing = 0
for col in ['VAL_TOT', 'VAL_SH', 'VAL_SP']:
    if col in sih_iam.columns:
        sih_iam[col] = sih_iam[col].fillna(0)

# Diagnósticos secundários — missing = sem comorbidade
diag_sec_cols = [c for c in sih_iam.columns if c.startswith('DIAGSEC')]
sih_iam[diag_sec_cols] = sih_iam[diag_sec_cols].fillna('SEM_DIAG')

# Categóricos — preserva ausência como categoria explícita
for col in ['RACA_COR', 'MARCA_UTI', 'NACIONAL', 'INFEHOSP',
            'FINANC', 'FAEC_TP', 'REGCT', 'NAT_JUR']:
    if col in sih_iam.columns:
        sih_iam[col] = sih_iam[col].fillna('NAO_INF')

# CID_MORTE — só tem valor em óbitos; '0000' = não aplicável
if 'CID_MORTE' in sih_iam.columns:
    sih_iam['CID_MORTE'] = sih_iam['CID_MORTE'].fillna('0000')

# CID_ASSO e DIAG_SECUN
for col in ['CID_ASSO', 'DIAG_SECUN']:
    if col in sih_iam.columns:
        sih_iam[col] = sih_iam[col].fillna('SEM_DIAG')

# ── Verifica missing residual ─────────────────────────────────────────────────
missing_residual = sih_iam.isnull().sum()
missing_residual = missing_residual[missing_residual > 0]

if len(missing_residual) == 0:
    print('✓ Nenhum missing residual nas colunas mantidas')
else:
    print(f'Missing residual em {len(missing_residual)} coluna(s):')
    pct = (missing_residual / len(sih_iam) * 100).round(1)
    print(pd.DataFrame({'n': missing_residual, 'pct': pct}).to_string())
    print('\n→ Verifique se alguma dessas colunas precisa de tratamento adicional')

In [ ]:
# ── Salva SIH limpo ───────────────────────────────────────────────────────────
sih_iam.to_parquet(INTERIM / 'sih_iam.parquet', index=False)

print(f'✓ Salvo: {INTERIM}/sih_iam.parquet')
print(f'  Linhas  : {len(sih_iam):,}')
print(f'  Colunas : {sih_iam.shape[1]}')
print(f'  Óbitos  : {sih_iam["MORTE"].sum():,}  ({sih_iam["MORTE"].mean():.2%})')

---
## 8. Checklist ETL

Antes de avançar para o CNES, confirme:

In [ ]:
checks = {
    'Base não vazia'                   : len(sih_iam) > 0,
    'Apenas CID I21'                   : sih_iam['DIAG_PRINC'].str.startswith('I21').all(),
    'Apenas adultos (>=18)'            : (sih_iam['IDADE_num'] >= 18).all(),
    'MORTE é binária sem nulos'        : sih_iam['MORTE'].isin([0, 1]).all() and
                                         sih_iam['MORTE'].isnull().sum() == 0,
    'dias_internacao entre 0 e 365'    : sih_iam['dias_internacao'].between(0, 365).all(),
    'CNES com 7 dígitos'               : sih_iam['CNES'].str.len().eq(7).all(),
    'N_AIH sem duplicatas'             : sih_iam['N_AIH'].nunique() == len(sih_iam),
    'Parquet gerado'                   : (INTERIM / 'sih_iam.parquet').exists(),
}

for descricao, passou in checks.items():
    print(f'  {"✅" if passou else "❌"}  {descricao}')

if all(checks.values()):
    print('\n🎉 SIH pronto — prossiga para o carregamento do CNES')
else:
    print('\n⚠️  Corrija os itens ❌ antes de continuar')